# DNA Encoding and Tokenization for Genomic Models

**Day 1 Morning - Session 1**

**Author:** Ikram Ullah, KAUST Bioinformatics Platform

---

## Welcome!

Before we can use deep learning models on DNA sequences, we need to convert nucleotide strings into numerical representations. This notebook covers the **foundational concepts** you'll use throughout the workshop.

---

## Learning Objectives

By completing this notebook, you will understand:

1. **Integer encoding** - The simplest DNA-to-number mapping
2. **One-hot encoding** - Binary vector representation
3. **Byte Pair Encoding (BPE)** - Learned subword tokenization (used by DNABERT-2)
4. **K-mer tokenization** - Fixed-length tokenization (used by Nucleotide Transformer)
5. **Loading models from Hugging Face Hub** - Accessing pre-trained genomic models

---

## Prerequisites

- Basic Python knowledge
- Understanding of DNA sequences (A, T, C, G nucleotides)
- PyTorch basics (tensors)

---

## The Big Picture

```
DNA String  →  Tokenization  →  Embeddings  →  Transformer  →  Predictions
  "ATCG"        [3,4,5,6]       [0.1,...]      [layers]        [0.9,0.1]
```

This notebook focuses on the first arrow: **DNA String → Tokenization**

---

## Setup: Import Required Packages

In [1]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel, AutoConfig

# Check PyTorch and GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

PyTorch version: 2.9.1+cu128
CUDA available: True
Using device: cuda


---

## Part 1: Basic DNA Encoding

### 1.1 Integer Encoding

The simplest encoding: each nucleotide becomes an integer index.

| Nucleotide | Integer |
|------------|--------:|
| A | 0 |
| T | 1 |
| C | 2 |
| G | 3 |

In [2]:
# Example DNA sequence
dna = "ATCGATCGATCG"

# Create a mapping: nucleotide -> integer
mapping = {'A': 0, 'T': 1, 'C': 2, 'G': 3}

# Convert each nucleotide to its integer representation
encoded = [mapping[base] for base in dna]

# Convert Python list to PyTorch tensor
tensor = torch.tensor(encoded)

# Display results
print(f"Original DNA:   {dna}")
print(f"Integer codes:  {encoded}")
print(f"PyTorch tensor: {tensor}")
print(f"Tensor shape:   {tensor.shape}  # (sequence_length,)")

Original DNA:   ATCGATCGATCG
Integer codes:  [0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3]
PyTorch tensor: tensor([0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3])
Tensor shape:   torch.Size([12])  # (sequence_length,)


### 1.2 One-Hot Encoding

One-hot encoding represents each nucleotide as a 4-dimensional binary vector:

| Nucleotide | One-Hot Vector |
|------------|----------------|
| A | [1, 0, 0, 0] |
| T | [0, 1, 0, 0] |
| C | [0, 0, 1, 0] |
| G | [0, 0, 0, 1] |

**Advantages:**
- No implicit ordering between nucleotides
- Works well with traditional ML models
- Easy to interpret

In [4]:
def one_hot_encode(seq):
    """
    Convert a DNA sequence to one-hot encoding.
    
    Args:
        seq: DNA string (e.g., "ATCG")
    
    Returns:
        Tensor of shape (seq_length, 4)
    """
    mapping = {'A': 0, 'T': 1, 'C': 2, 'G': 3}
    one_hot = torch.zeros(len(seq), 4)
    for i, base in enumerate(seq):
        one_hot[i, mapping[base]] = 1
    return one_hot

# Encode a single sequence
encoded = one_hot_encode("ATCAG")
print("One-hot encoding of 'ATCG':")
print(encoded)
print(f"Shape: {encoded.shape}  # (seq_length=4, nucleotides=4)")

# Batch multiple sequences
print("\n--- Batching multiple sequences ---")
sequences = ["ATCG", "GCTA"]
batch = torch.stack([one_hot_encode(s) for s in sequences])
print(f"Batch shape: {batch.shape}  # (batch=2, seq_len=4, nucleotides=4)")

One-hot encoding of 'ATCG':
tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [1., 0., 0., 0.],
        [0., 0., 0., 1.]])
Shape: torch.Size([5, 4])  # (seq_length=4, nucleotides=4)

--- Batching multiple sequences ---
Batch shape: torch.Size([2, 4, 4])  # (batch=2, seq_len=4, nucleotides=4)


---

## Part 2: Advanced Tokenization

Modern genomic models use more sophisticated tokenization methods that learn patterns from data.

### 2.1 Byte Pair Encoding (BPE)
**Byte Pair Encoding (BPE)** is a subword tokenization algorithm used by genomic language models such as **DNABERT‑2**.

#### How BPE Works

1. Start with individual characters as the initial vocabulary (A, T, C, G)
2. Count all adjacent token pairs in the training corpus and find the most frequent pair.
3. Merge that pair into a new token, add it to the vocabulary, and replace all occurrences of the pair in the corpus.
4. Repeat this merge step until the target vocabulary size (or number of merges) is reached.

**Example**: If `"AT"` appears frequently, BPE creates a new token `"AT"`. Later, if `"ATG"` is common, it becomes another token.

#### Why BPE for Genomics?

| Advantage | Description |
|-----------|-------------|
| **Variable-length tokens** | Unlike fixed k-mers, BPE learns optimal token lengths |
| **Efficient vocabulary** | More compact than enumerating all k-mers |
| **Handles rare patterns** | Can fall back to smaller subwords |


#### Example (illustrative, not actual BPE output):
```
Sequence: "ATCGATCGATCGATCG"
```
* Small vocab (few merges): might break into shorter chunks, e.g. 6 tokens
* Large vocab (more merges): might use longer learned tokens, e.g. 2 tokens like `ATCGATCG` and `ATCGATCG`

**Smaller vocabulary**

* More flexible: can compose rare or novel patterns from shorter subwords.
* Typically more tokens per sequence → longer effective context, higher compute per example.
* Each token is seen more often, so embeddings are trained on more occurrences.

**Larger vocabulary**
* More compact: fewer tokens per sequence → cheaper attention, faster inference.
* Some tokens are rare, so their embeddings are trained on fewer examples and can be poorly estimated.
* Very large vocab can introduce redundancy and does not guarantee better generalization.

In [6]:
# Load DNABERT-2 tokenizer (uses BPE)
bpe_tokenizer = AutoTokenizer.from_pretrained(
    "zhihan1996/DNABERT-2-117M", 
    trust_remote_code=True
)

# Example DNA sequence
dna_sequence = "ATCGATCGATCGATCG"

# Tokenize the sequence
tokens = bpe_tokenizer.tokenize(dna_sequence)
token_ids = bpe_tokenizer.encode(dna_sequence)

print("BPE Tokenization (DNABERT-2)")
print("=" * 50)
print(f"Input sequence:   {dna_sequence}")
print(f"Sequence length:  {len(dna_sequence)} nucleotides")
print(f"\nTokens:           {tokens}")
print(f"Number of tokens: {len(tokens)}")
print(f"\nToken IDs:        {token_ids}")
print(f"\nVocabulary size:  {bpe_tokenizer.vocab_size:,}")

BPE Tokenization (DNABERT-2)
Input sequence:   ATCGATCGATCGATCG
Sequence length:  16 nucleotides

Tokens:           ['A', 'TCGA', 'TCGA', 'TCGA', 'TC', 'G']
Number of tokens: 6

Token IDs:        [1, 5, 359, 359, 359, 16, 7, 2]

Vocabulary size:  4,096


### 2.2 K-mer Tokenization

**K-mer tokenization** splits a DNA sequence into contiguous substrings of length k 
    * often overlapping, e.g. ATCGAT → 3-mers: ATC, TCG, CGA, GAT

For 6-mers over {A,T,C,G}, the theoretical vocabulary size is 4^6 = 4,096, plus special tokens.


| Aspect       | BPE                                      | K-mer                                         |
|--------------|-------------------------------------------|-----------------------------------------------|
| Token length | Variable (data-learned)                  | Fixed (e.g., 6)                               |
| Vocabulary   | Configurable                             | ~4^k (given k, alphabet fixed)                |
| Flexibility  | High (can form many variable-length units) | Lower (fixed local window; great for local motifs) |


In [7]:
# Load Nucleotide Transformer tokenizer (uses k-mers)
kmer_tokenizer = AutoTokenizer.from_pretrained(
    "InstaDeepAI/nucleotide-transformer-v2-50m-3mer-multi-species",
    trust_remote_code=True
)

# Same DNA sequence
dna_sequence = "ATCGATCGATCGATCG"

# Tokenize with k-mer tokenizer
kmer_tokens = kmer_tokenizer.tokenize(dna_sequence)
kmer_ids = kmer_tokenizer.encode(dna_sequence)

print("K-mer Tokenization (Nucleotide Transformer)")
print("=" * 50)
print(f"Input sequence:   {dna_sequence}")
print(f"\nTokens:           {kmer_tokens}")
print(f"Number of tokens: {len(kmer_tokens)}")
print(f"\nToken IDs:        {kmer_ids}")
print(f"\nVocabulary size:  {kmer_tokenizer.vocab_size:,}")

K-mer Tokenization (Nucleotide Transformer)
Input sequence:   ATCGATCGATCGATCG

Tokens:           ['ATC', 'GAT', 'CGA', 'TCG', 'ATC', 'G']
Number of tokens: 6

Token IDs:        [3, 12, 55, 50, 33, 12, 73]

Vocabulary size:  75


In [8]:
# Side-by-side comparison
print("Comparison: BPE vs K-mer Tokenization")
print("=" * 50)
print(f"Input: {dna_sequence} ({len(dna_sequence)} bp)\n")

bpe_tokens = bpe_tokenizer.tokenize(dna_sequence)
kmer_tokens = kmer_tokenizer.tokenize(dna_sequence)

print(f"BPE (DNABERT-2):")
print(f"  Tokens: {bpe_tokens}")
print(f"  Count:  {len(bpe_tokens)}")

print(f"\nK-mer (Nucleotide Transformer):")
print(f"  Tokens: {kmer_tokens}")
print(f"  Count:  {len(kmer_tokens)}")

print("\n--- Key Difference ---")
print("BPE: Variable-length tokens learned from data patterns")
print("K-mer: Fixed-length tokens (each = k nucleotides)")

Comparison: BPE vs K-mer Tokenization
Input: ATCGATCGATCGATCG (16 bp)

BPE (DNABERT-2):
  Tokens: ['A', 'TCGA', 'TCGA', 'TCGA', 'TC', 'G']
  Count:  6

K-mer (Nucleotide Transformer):
  Tokens: ['ATC', 'GAT', 'CGA', 'TCG', 'ATC', 'G']
  Count:  6

--- Key Difference ---
BPE: Variable-length tokens learned from data patterns
K-mer: Fixed-length tokens (each = k nucleotides)


---

## Part 3: Loading Models from Hugging Face Hub

The [Hugging Face Hub](https://huggingface.co/models) hosts thousands of pre-trained models. For genomics, the key classes are:

- **AutoTokenizer**: Converts DNA strings → token IDs
- **AutoModel**: Loads the transformer encoder
- **AutoConfig**: Access model configuration

### 3.1 Load DNABERT-2

In [9]:
from transformers import AutoTokenizer, AutoModel, BertConfig
from IPython.display import display, Markdown

# Model identifier on Hugging Face Hub
model_name = "zhihan1996/DNABERT-2-117M"

# Load the three key components
print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
config = BertConfig.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, trust_remote_code=True, config=config)

# Count parameters
params = sum(p.numel() for p in model.parameters())

print(f"\nDNABERT-2 loaded successfully!")
print(f"  Architecture:    BERT + ALiBi")
print(f"  Tokenizer:       BPE (vocab size: {tokenizer.vocab_size:,})")
print(f"  Hidden size:     {config.hidden_size}")
print(f"  Layers:          {config.num_hidden_layers}")
print(f"  Attention heads: {config.num_attention_heads}")
print(f"  Parameters:      {params:,} (~{params/1e6:.0f}M)")

Loading zhihan1996/DNABERT-2-117M...


Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



DNABERT-2 loaded successfully!
  Architecture:    BERT + ALiBi
  Tokenizer:       BPE (vocab size: 4,096)
  Hidden size:     768
  Layers:          12
  Attention heads: 12
  Parameters:      117,068,544 (~117M)


### 3.2 Load Nucleotide Transformer

In [ ]:
# Nucleotide Transformer model (using 50m 3-mer model which loads reliably)
nt_name = "InstaDeepAI/nucleotide-transformer-v2-50m-3mer-multi-species"

print(f"Loading {nt_name}...")
nt_tokenizer = AutoTokenizer.from_pretrained(nt_name, trust_remote_code=True)
nt_config = AutoConfig.from_pretrained(nt_name, trust_remote_code=True)
nt_model = AutoModel.from_pretrained(nt_name, trust_remote_code=True, config=nt_config)

# Count parameters
nt_params = sum(p.numel() for p in nt_model.parameters())

print(f"\nNucleotide Transformer loaded successfully!")
print(f"  Architecture:    ESM-based Transformer")
print(f"  Tokenizer:       K-mer (vocab size: {nt_tokenizer.vocab_size:,})")
print(f"  Hidden size:     {nt_config.hidden_size}")
print(f"  Layers:          {nt_config.num_hidden_layers}")
print(f"  Attention heads: {nt_config.num_attention_heads}")
print(f"  Parameters:      {nt_params:,} (~{nt_params/1e6:.0f}M)")

### 3.3 Running a Forward Pass

Let's tokenize a DNA sequence and pass it through a model to get embeddings.

In [ ]:
# Move model to device
nt_model = nt_model.to(device).eval()

# Example sequence
dna_sequence = "ATCGATCGATCGATCGATCGATCGATCGATCG"

# Tokenize
inputs = nt_tokenizer(
    dna_sequence, 
    return_tensors="pt",
    padding=True,
    truncation=True
).to(device)

print("Tokenized input:")
print(f"  input_ids shape: {inputs['input_ids'].shape}")
print(f"  input_ids: {inputs['input_ids'][0].tolist()}")

# Forward pass
with torch.no_grad():
    outputs = nt_model(**inputs)

# Get embeddings
embeddings = outputs.last_hidden_state

print(f"\nOutput embeddings:")
print(f"  Shape: {embeddings.shape}  # (batch, seq_len, hidden_dim)")

# Mean pooling to get single vector per sequence
sequence_embedding = embeddings.mean(dim=1)
print(f"  After mean pooling: {sequence_embedding.shape}  # (batch, hidden_dim)")

---

## Summary

In this notebook, you learned:

| Concept | Description |
|---------|-------------|
| **Integer encoding** | Simple A=0, T=1, C=2, G=3 mapping |
| **One-hot encoding** | Binary vectors, no implicit ordering |
| **BPE tokenization** | Variable-length learned tokens (DNABERT-2) |
| **K-mer tokenization** | Fixed-length tokens (Nucleotide Transformer) |
| **Loading models** | Using AutoTokenizer and AutoModel from HF Hub |
| **Forward pass** | Tokenize → Model → Embeddings |

**Next**: In the following notebook, we'll explore the **Nucleotide Transformer architecture** in detail.